In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

// 🔇 Silenciar TODOS los logs de Spark (log4j 2 — el que usa Spark 4)
import org.apache.logging.log4j.{Level, LogManager}
import org.apache.logging.log4j.core.config.Configurator

Configurator.setRootLevel(Level.ERROR)
Configurator.setLevel("org",         Level.ERROR)
Configurator.setLevel("org.apache",  Level.ERROR)
Configurator.setLevel("org.apache.spark", Level.ERROR)
Configurator.setLevel("org.sparkproject", Level.ERROR)
Configurator.setLevel("akka",        Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Ejercicios")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

val sc = spark.sparkContext

// Volvemos a silenciar tras crear la sesión, por si Spark reinicia el logger
Configurator.setRootLevel(Level.ERROR)

println(s"✅ Entorno listo — Spark ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 02:33:11 INFO SparkContext: Running Spark version 4.1.1
26/04/28 02:33:11 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/28 02:33:11 INFO SparkContext: Java version 17.0.18+8
26/04/28 02:33:12 INFO ResourceUtils: ==============================================================
26/04/28 02:33:12 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/28 02:33:12 INFO ResourceUtils: ==============================================================
26/04/28 02:33:12 INFO SparkContext: Submitted application: Ejercicios
26/04/28 02:33:12 INFO SecurityManager: Changing view acls to: gre
26/04/28 02:33:12 INFO SecurityManager: Changing modify acls to: gre
26/04/28 02:33:12 INFO SecurityManager: Changing view acls groups to: gre
26/04/28 02:33:12 INFO SecurityManager: Changing modify acls groups to: gre
26/04/28 02:33:12 INFO SecurityManager: SecurityManager: authentication disable

✅ Entorno listo — Spark 4.1.1


import $ivy.$
import $ivy.$
import org.apache.logging.log4j.{Level, LogManager}
import org.apache.logging.log4j.core.config.Configurator
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@45060578
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@55b8e427

Ejercicio 1 — RDD de temperaturas

In [2]:
val temperaturas = List(22.5, 18.0, 35.1, 12.3, 28.7, 9.4, 31.0, 25.5, 17.2, 33.8)
val tempsRDD = sc.parallelize(temperaturas)

println(s"Tipo del RDD: ${tempsRDD.getClass.getSimpleName}")
println(s"Número de particiones: ${tempsRDD.getNumPartitions}")

Tipo del RDD: ParallelCollectionRDD
Número de particiones: 16


temperaturas: List[Double] = List(
  22.5,
  18.0,
  35.1,
  12.3,
  28.7,
  9.4,
  31.0,
  25.5,
  17.2,
  33.8
)
tempsRDD: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:2

Ejercicio 2 — Filtrar entre 20 y 30

In [3]:
val entre20y30 = tempsRDD.filter(t => t > 20).filter(t => t < 30)
println(s"Temperaturas entre 20 y 30 grados: ${entre20y30.collect().toList}")

Temperaturas entre 20 y 30 grados: List(22.5, 28.7, 25.5)


entre20y30: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[2] at filter at cmd3.sc:1

Ejercicio 3 — Celsius → Fahrenheit


In [4]:
val fahrenheit = tempsRDD.map(c => c * 9.0 / 5.0 + 32.0)
val resultado = fahrenheit.collect().map(f => f"$f%.1f")
println("Temperaturas en Fahrenheit:")
println(resultado.mkString(", "))

Temperaturas en Fahrenheit:
72,5, 64,4, 95,2, 54,1, 83,7, 48,9, 87,8, 77,9, 63,0, 92,8


fahrenheit: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[3] at map at cmd4.sc:1
resultado: Array[String] = Array(
  "72,5",
  "64,4",
  "95,2",
  "54,1",
  "83,7",
  "48,9",
  "87,8",
  "77,9",
  "63,0",
  "92,8"
)

Ejercicio 4 — Estadística básica

In [5]:
val precios = sc.parallelize(List(49.99, 12.50, 199.00, 7.99, 89.90, 34.99, 149.95, 22.00))

println(s"Número de artículos: ${precios.count()}")
println(s"Precio más alto:  ${precios.max()}")
println(s"Precio más bajo:  ${precios.min()}")

Número de artículos: 8
Precio más alto:  199.0
Precio más bajo:  7.99


precios: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[4] at parallelize at cmd5.sc:1

Ejercicio 5 — Lazy evaluation (en DOS celdas)

In [6]:
val numeros = sc.parallelize(1 to 100)
val multiplosde7 = numeros.filter(n => n % 7 == 0)
val dobles = multiplosde7.map(n => n * 2)
println("Celda A ejecutada — ¿Hay un nuevo job en el Spark UI?")

Celda A ejecutada — ¿Hay un nuevo job en el Spark UI?


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[5] at parallelize at cmd6.sc:1
multiplosde7: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[6] at filter at cmd6.sc:2
dobles: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[7] at map at cmd6.sc:3

In [7]:
val resultadoLazy = dobles.collect()
println("Celda B ejecutada — ¿Y ahora en el Spark UI?")
println(s"Resultado: ${resultadoLazy.mkString(", ")}")

Celda B ejecutada — ¿Y ahora en el Spark UI?
Resultado: 14, 28, 42, 56, 70, 84, 98, 112, 126, 140, 154, 168, 182, 196


resultadoLazy: Array[Int] = Array(
  14,
  28,
  42,
  56,
  70,
  84,
  98,
  112,
  126,
  140,
  154,
  168,
  182,
  196
)

Ejercicio 6 — Pipeline de nombres

In [8]:
val empleados = sc.parallelize(List(
  "ana garcía", "pedro López", "alba martínez", "carlos ruiz",
  "adriana vega", "beatriz soler", "antonio mora", "sara jiménez"
))

val resultadoEmp = empleados
  .map(_.toUpperCase)
  .filter(_.startsWith("A"))
  .sortBy((s: String) => s)   // 👈 tipado explícito en lugar de identity
  .collect()

println("Empleados cuyo nombre empieza por A (en mayúsculas):")
resultadoEmp.foreach(println)

Empleados cuyo nombre empieza por A (en mayúsculas):
ADRIANA VEGA
ALBA MARTÍNEZ
ANA GARCÍA
ANTONIO MORA


empleados: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[8] at parallelize at cmd8.sc:1
resultadoEmp: Array[String] = Array(
  "ADRIANA VEGA",
  "ALBA MARTÍNEZ",
  "ANA GARCÍA",
  "ANTONIO MORA"
)

Ejercicio 7 — flatMap


In [9]:
val frases = sc.parallelize(List(
  "spark procesa datos a gran velocidad",
  "scala es el lenguaje nativo de spark",
  "los datos se procesan en memoria ram",
  "hadoop almacena datos en disco hdfs"
))

val palabras = frases.flatMap(_.split(" "))
println(s"Total de palabras en todas las frases: ${palabras.count()}")

Total de palabras en todas las frases: 26


frases: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[16] at parallelize at cmd9.sc:1
palabras: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[17] at flatMap at cmd9.sc:8

Ejercicio 8 — reduce sin max()


In [10]:
val ventasDiarias = sc.parallelize(List(1520.0, 890.5, 2340.0, 670.0, 1890.75, 3100.0, 450.25))
val maxVenta = ventasDiarias.reduce((a, b) => if (a > b) a else b)
println(s"Venta diaria más alta: $maxVenta €")

Venta diaria más alta: 3100.0 €


ventasDiarias: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[18] at parallelize at cmd10.sc:1
maxVenta: Double = 3100.0

Ejercicio 9 — Mayores de edad

In [11]:
val edades = sc.parallelize(List(
  14, 23, 17, 35, 16, 28, 42, 15, 19, 31, 13, 25, 18, 22, 16, 45, 20, 12
))

val total = edades.count()
val mayores = edades.filter(_ >= 18).count()
val porcentaje = mayores.toDouble / total * 100

println(f"Total visitantes:    $total")
println(f"Mayores de edad:     $mayores")
println(f"Porcentaje:          $porcentaje%.1f%%")

Total visitantes:    18
Mayores de edad:     11
Porcentaje:          61,1%


edades: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[19] at parallelize at cmd11.sc:1
total: Long = 18L
mayores: Long = 11L
porcentaje: Double = 61.111111111111114

Ejercicio 10 — Hashtags

In [12]:
val publicaciones = sc.parallelize(List(
  "hoy aprendemos #spark con #scala en clase",
  "me encanta #scala es un lenguaje potente",
  "procesando big data con #spark y #hadoop",
  "#spark es más rápido que #hadoop en memoria",
  "aprendiendo #scala y #spark juntos hoy"
))

val hashtags = publicaciones
  .flatMap(_.split(" "))
  .filter(_.startsWith("#"))
  .map(h => (h, 1))
  .reduceByKey(_ + _)
  .sortBy({ case (_, c) => c }, ascending = false)
  .collect()

println("Frecuencia de hashtags:")
hashtags.foreach { case (h, c) => println(f"  $h%-8s → $c veces") }

Frecuencia de hashtags:
  #spark   → 4 veces
  #scala   → 3 veces
  #hadoop  → 2 veces


publicaciones: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[21] at parallelize at cmd12.sc:1
hashtags: Array[(String, Int)] = Array(
  ("#spark", 4),
  ("#scala", 3),
  ("#hadoop", 2)
)

## 🏢 Caso de Empresa MediaStream

Tarea 1 — Cargar datos


In [13]:
val reproducciones = List(
  ("U001", "La Casa de Papel",         "Serie",        47),
  ("U002", "Dune: Parte Dos",          "Película",     95),
  ("U003", "Cosmos: Mundos Posibles",  "Documental",   42),
  ("U001", "Dune: Parte Dos",          "Película",    107),
  ("U004", "La Casa de Papel",         "Serie",        52),
  ("U005", "El Problema de los 3 Cuerpos", "Serie",   58),
  ("U002", "Cosmos: Mundos Posibles",  "Documental",   38),
  ("U006", "La Casa de Papel",         "Serie",        49),
  ("U003", "El Problema de los 3 Cuerpos", "Serie",   61),
  ("U007", "Dune: Parte Dos",          "Película",     88),
  ("U005", "La Casa de Papel",         "Serie",        44),
  ("U008", "Cosmos: Mundos Posibles",  "Documental",   55),
  ("U004", "El Problema de los 3 Cuerpos", "Serie",   63),
  ("U009", "Dune: Parte Dos",          "Película",    112),
  ("U006", "Cosmos: Mundos Posibles",  "Documental",   29),
  ("U010", "La Casa de Papel",         "Serie",        51),
  ("U007", "El Problema de los 3 Cuerpos", "Serie",   57),
  ("U008", "La Casa de Papel",         "Serie",        46),
  ("U001", "Cosmos: Mundos Posibles",  "Documental",   33),
  ("U009", "La Casa de Papel",         "Serie",        48),
  ("U010", "Dune: Parte Dos",          "Película",    102),
  ("U002", "El Problema de los 3 Cuerpos", "Serie",   60),
  ("U003", "Dune: Parte Dos",          "Película",     91),
  ("U011", "Cosmos: Mundos Posibles",  "Documental",   47),
  ("U012", "La Casa de Papel",         "Serie",        53)
)

val reprosRDD = sc.parallelize(reproducciones)

println("✅ Entorno MediaStream listo")
println(s"   Spark version : ${spark.version}")
println(s"   App name      : ${sc.appName}")
println(s"   Particiones   : ${reprosRDD.getNumPartitions}")
println(s"   Registros     : ${reprosRDD.count()}")

✅ Entorno MediaStream listo
   Spark version : 4.1.1
   App name      : Ejercicios
   Particiones   : 16
   Registros     : 25


reproducciones: List[(String, String, String, Int)] = List(
  ("U001", "La Casa de Papel", "Serie", 47),
  ("U002", "Dune: Parte Dos", "Película", 95),
  ("U003", "Cosmos: Mundos Posibles", "Documental", 42),
  ("U001", "Dune: Parte Dos", "Película", 107),
  ("U004", "La Casa de Papel", "Serie", 52),
  ("U005", "El Problema de los 3 Cuerpos", "Serie", 58),
  ("U002", "Cosmos: Mundos Posibles", "Documental", 38),
  ("U006", "La Casa de Papel", "Serie", 49),
  ("U003", "El Problema de los 3 Cuerpos", "Serie", 61),
  ("U007", "Dune: Parte Dos", "Película", 88),
  ("U005", "La Casa de Papel", "Serie", 44),
  ("U008", "Cosmos: Mundos Posibles", "Documental", 55),
  ("U004", "El Problema de los 3 Cuerpos", "Serie", 63),
  ("U009", "Dune: Parte Dos", "Película", 112),
  ("U006", "Cosmos: Mundos Posibles", "Documental", 29),
  ("U010", "La Casa de Papel", "Serie", 51),
  ("U007", "El Problema de los 3 Cuerpos", "Serie", 57),
  ("U008", "La Casa de Papel", "Serie", 46),
  ("U001", "Cosmos: Mund

Tarea 2 — Reproducciones completas (>45 min)

In [14]:
val totalRepros  = reprosRDD.count()
val completas    = reprosRDD.filter { case (_, _, _, min) => min > 45 }.count()
val porcentajeRC = completas.toDouble / totalRepros * 100

println("── Reproducciones completas (> 45 min) ──")
println(s"Total de reproducciones ayer : $totalRepros")
println(s"Reproducciones completas     : $completas")
println(f"Porcentaje completadas       : $porcentajeRC%.1f%%")

── Reproducciones completas (> 45 min) ──
Total de reproducciones ayer : 25
Reproducciones completas     : 20
Porcentaje completadas       : 80,0%


totalRepros: Long = 25L
completas: Long = 20L
porcentajeRC: Double = 80.0

Tarea 3 — Minutos por género

In [15]:
val porGenero = reprosRDD
  .map { case (_, _, genero, min) => (genero, min) }
  .reduceByKey(_ + _)
  .sortBy({ case (_, m) => m }, ascending = false)
  .collect()

println("── Minutos totales por género ──")
porGenero.foreach { case (g, m) => println(f"$g%-12s → $m minutos") }

── Minutos totales por género ──
Serie        → 689 minutos
Película     → 595 minutos
Documental   → 244 minutos


porGenero: Array[(String, Int)] = Array(
  ("Serie", 689),
  ("Película", 595),
  ("Documental", 244)
)

Tarea 4 — Contenido más popular

In [16]:
val porTitulo = reprosRDD
  .map { case (_, titulo, _, _) => (titulo, 1) }
  .reduceByKey(_ + _)

val (tituloTop, reprosTop) = porTitulo.reduce { (a, b) =>
  if (a._2 > b._2) a else b
}

val usuariosUnicos = reprosRDD
  .filter { case (_, t, _, _) => t == tituloTop }
  .map { case (u, _, _, _) => u }
  .distinct()
  .count()

println("── Contenido más popular ──")
println(s"Título más visto         : $tituloTop")
println(s"Número de reproducciones : $reprosTop")
println(s"Usuarios únicos que lo vieron : $usuariosUnicos")

── Contenido más popular ──
Título más visto         : La Casa de Papel
Número de reproducciones : 8
Usuarios únicos que lo vieron : 8


porTitulo: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[41] at reduceByKey at cmd16.sc:3
tituloTop: String = "La Casa de Papel"
reprosTop: Int = 8
usuariosUnicos: Long = 8L

🚀 Pregunta adicional — Usuario más activo


In [17]:
val (usuarioTop, minTop) = reprosRDD
  .map { case (u, _, _, m) => (u, m) }
  .reduceByKey(_ + _)
  .reduce { (a, b) => if (a._2 > b._2) a else b }

println("── Usuario más activo del día ──")
println(s"Usuario : $usuarioTop")
println(s"Minutos totales vistos : $minTop")

── Usuario más activo del día ──
Usuario : U003
Minutos totales vistos : 194


usuarioTop: String = "U003"
minTop: Int = 194